# Introduction
## Barrier Option Pricing using Binomial Tree

### Group Members:
1. Yu Chen
2. Laaya Fani
3. Mitu Mevada

### Objective
In this Project, we price an Up-and-Out Barrier Call Option using a Binomial tree model.

### Model:
- Binomial Tree Model

### Assumptions:
- Underlying aaset: S&P 500
- Constant Volatility
- Constant interest rate

### Key Feature
The Option becomes worthless if the stock price crosses the barrier level.

In [56]:
import numpy as np

### Input Market Parameters

In [57]:
S0 = 7408.0      # S&P 500 Spot Price
k = 7408.0       #Strike Price (At the money)
B = 8800.0      # Adjusted Barrier Level (Up-and-out)
sigma = 0.1843   # Volatility (18.43%)
r = 0.0361       # Risk-free rate (3.61%)
q = 0.0106       # Dividend yield (1.06%)
T = 1.0          # Time to maturity (1 year)
N = 1000         # Number of time steps 


In [58]:
print(S0, k, B, sigma, r, q, T, N)

7408.0 7408.0 8800.0 0.1843 0.0361 0.0106 1.0 1000


### Calculate Tree Constants

In [59]:
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1 / u  # Down factor
p = (np.exp((r - q) * dt) - d) / (u - d)  # Risk-neutral probability
discount_factor = np.exp(-r * dt) 

### Initialize Stock Prices at Expiration (Step N)

In [60]:
stock_prices = S0 * (u ** np.arange(N, -1, -1)) * (d ** np.arange(0, N + 1))  # Stock price at maturity

### Calculate Option Payoffs at Expiration 

In [61]:
option_values = np.maximum(stock_prices - k, 0)  # Call option payoff at maturity
option_values[stock_prices >= B] = 0  # Set option value to zero if barrier is breached 

### Backward Induction Loop

In [62]:
discount_factor = np.exp(-r * dt)

for i in range(N - 1, -1, -1):
    
    # Compute stock prices at time i
    stock_prices = S0 * (u ** np.arange(i, -1, -1)) * (d ** np.arange(0, i + 1))

    # Update option values 
    option_values[:i+1] = discount_factor * (
        p * option_values[1:i+2] + (1 - p) * option_values[:i+1]
    )

    # Apply barrier condition
    option_values[:i+1][stock_prices >= B] = 0.0


### Output the Final Price

In [63]:
option_price = option_values[ 0]
print(f"The calculated price of the up-and-out barrier call option is: {option_price:.2f}")

The calculated price of the up-and-out barrier call option is: 83.89


### Monte-Carlo Simulation

In [10]:
import numpy as np

### Input Market Parameters (Exact same as the Binomial Tree)

In [ ]:

S0 = 7408.0       # S&P 500 Spot Price
K = 7408.0        # Strike Price
B = 8800.0        # Barrier Level
sigma = 0.1843    # Volatility (18.43%)
r = 0.0361        # Risk-Free Rate (3.61%)
q = 0.0106        # Dividend Yield (1.06%)
T = 1.0           # Time to Expiration (1 Year)

### Simulation Settings

In [ ]:

M = 50000         # Number of random paths to generate
N = 252           # Number of time steps per path (252 trading days in a year)
dt = T / N        # Time increment for each day

### Calculation of Match Constants for Path Generation

In [ ]:

drift = (r - q - 0.5 * sigma**2) * dt
volatility_effect = sigma * np.sqrt(dt)

### Generation of Random Market Noise

In [ ]:

Z = np.random.standard_normal((N, M))

### Build the S&P 500 Price Matrix

In [ ]:

S = np.zeros((N + 1, M))
S[0] = S0  # Day 0 for all paths is today's spot price (7,408)

for t in range(1, N + 1):
    S[t] = S[t-1] * np.exp(drift + volatility_effect * Z[t-1])

### Application of the Up-and-Out Barrier Rule

In [ ]:

# Find the highest price the S&P 500 reached during the year for each path
path_maximums = np.max(S, axis=0)

# Calculate the standard final option payoff at expiration: Max(S_final - K, 0)
final_prices = S[-1]
payoffs = np.maximum(final_prices - K, 0)

# The Tripwire: If a path's maximum price hit or exceeded the barrier, its payoff is $0
payoffs[path_maximums >= B] = 0.0

### Averaged and Discounted back to Today

In [ ]:

average_payoff = np.mean(payoffs)
mc_option_price = np.exp(-r * T) * average_payoff

print(f"The Monte Carlo simulated option price is: ${mc_option_price:.2f}")

The Monte Carlo simulated option price is: $95.76


### Conclusion and Comparison

- Both models successfully capture the massive premium discount of an exotic barrier option compared to a standard vanilla call option (which would cost roughly $550). The Binomial Tree checks for the barrier at rigid, discrete nodes, while the Monte Carlo simulation tracks fluid, continuous daily price paths, making it more sensitive to the knock-out tripwire over a long 1-year horizon.
